# Script para unificar archivo con información historica de dosis de vacunación con el archivo con la información de casos confirmados de sarampion.

In [1]:
import pandas as pd
import numpy as np

# 1. Cargar el boletín epidemiológico
print("Cargando boletín epidemiológico...")
df_boletin = pd.read_csv('../data/processed/boletin_epidemiologico_2000-2025_unificado.csv')

# 2. Filtrar solo Sarampión y Casos_Semana_Actual
df_sarampion = df_boletin[
    (df_boletin['Enfermedad'] == 'Sarampión') & 
    (df_boletin['Variable'] == 'Casos_Semana_Actual')
].copy()

# Limpiar valores no numéricos (como '----') y convertir a numérico
df_sarampion['Valor'] = pd.to_numeric(df_sarampion['Valor'], errors='coerce').fillna(0)

print(f"Registros de sarampión (casos semanales): {len(df_sarampion)}")

# 3. Convertir semanas a meses (4-5 semanas por mes)
# Crear fecha a partir del año y semana epidemiológica
df_sarampion['fecha'] = pd.to_datetime(
    df_sarampion['Año'].astype(str) + '-W' + 
    df_sarampion['Semana'].astype(str).str.zfill(2) + '-1', 
    format='%Y-W%W-%w'
)

df_sarampion['mes'] = df_sarampion['fecha'].dt.month

# 4. Agrupar por año-mes y sumar casos
df_sarampion_mensual = df_sarampion.groupby(['Año', 'mes']).agg({
    'Valor': 'sum'
}).reset_index()

df_sarampion_mensual.columns = ['anio', 'mes', 'casos_sarampion']

print(f"Registros mensuales de sarampión: {len(df_sarampion_mensual)}")

# 5. Filtrar años 2010-2023
df_sarampion_2010_2023 = df_sarampion_mensual[
    (df_sarampion_mensual['anio'] >= 2010) & 
    (df_sarampion_mensual['anio'] <= 2023)
].copy()

print(f"Registros 2010-2023: {len(df_sarampion_2010_2023)}")

# 6. Cargar datos de vacunación
print("\nCargando datos de vacunación...")
df_vacunas = pd.read_csv('../data/processed/dosis_vacuna_2010-2023_unificado.csv')

print(f"Registros de vacunación: {len(df_vacunas)}")

# 7. Unir ambos datasets
df_unificado = pd.merge(
    df_vacunas,
    df_sarampion_2010_2023,
    on=['anio', 'mes'],
    how='left'
)

# Rellenar casos de sarampión con 0 si no hay datos (aunque deberían estar todos)
df_unificado['casos_sarampion'] = df_unificado['casos_sarampion'].fillna(0).astype(int)

# 8. Ordenar por año y mes
df_unificado = df_unificado.sort_values(['anio', 'mes']).reset_index(drop=True)

# 9. Guardar dataset unificado
output_file = '../data/processed/dataset_sarampion_vacunas_2010-2023.csv'
df_unificado.to_csv(output_file, index=False)

print(f"\n{'='*60}")
print(f"DATASET UNIFICADO CREADO: {output_file}")
print(f"{'='*60}")
print(f"Período: 2010-2023")
print(f"Total registros: {len(df_unificado)}")
print(f"Total dosis administradas: {df_unificado['total_dosis'].sum():,}")
print(f"Total casos de sarampión: {df_unificado['casos_sarampion'].sum():,}")
print(f"\nColumnas: {list(df_unificado.columns)}")

print("\nPrimeros 10 registros:")
print(df_unificado.head(10))

print("\nÚltimos 10 registros:")
print(df_unificado.tail(10))

print("\nEstadísticas descriptivas:")
print(df_unificado[['total_dosis', 'casos_sarampion']].describe())

Cargando boletín epidemiológico...
Registros de sarampión (casos semanales): 1275
Registros mensuales de sarampión: 294
Registros 2010-2023: 156

Cargando datos de vacunación...
Registros de vacunación: 168

DATASET UNIFICADO CREADO: ../data/processed/dataset_sarampion_vacunas_2010-2023.csv
Período: 2010-2023
Total registros: 168
Total dosis administradas: 52,974,270
Total casos de sarampión: 380

Columnas: ['mes', 'anio', 'total_dosis', 'casos_sarampion']

Primeros 10 registros:
   mes  anio  total_dosis  casos_sarampion
0    1  2010       180830                0
1    2  2010       183377                0
2    3  2010       242572                0
3    4  2010       216429                0
4    5  2010       174559                0
5    6  2010       411045                0
6    7  2010       229487               81
7    8  2010       254698               25
8    9  2010       206088                0
9   10  2010       396236               15

Últimos 10 registros:
     mes  anio  tot